## Cell 1 - Note

In [1]:
# ============================================================
# 04_transfer_finetune.ipynb
# Target-domain transfer fine-tuning for hybrid AoA + amplitude model
# ============================================================
#
# Inputs:
#   - pred_support.npz
#   - pred_query.npz
#   - hybrid_model_best.keras
#
# Outputs:
#   - hybrid_transfer_best.keras
#   - hybrid_transfer_final.keras
#   - transfer_history.json
# ============================================================

## Cell 2 - Import

In [2]:
import os
import json
import math
import random
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

2026-03-21 19:50:23.281861: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-21 19:50:23.288330: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774122623.295949  534921 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774122623.298170  534921 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774122623.303932  534921 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## Cell 3 - Config

In [3]:
# ============================================================
# CONFIG
# ============================================================

DATA_ROOT = Path("/home/tonyliao/Location_AMP")   # change this
BUILD_DIR = DATA_ROOT / "dataset_build_hybrid"
TRAIN_DIR = DATA_ROOT / "hybrid_train_runs"
OUT_DIR   = DATA_ROOT / "hybrid_transfer_runs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PRED_SUPPORT_NPZ = BUILD_DIR / "pred_support.npz"
PRED_QUERY_NPZ   = BUILD_DIR / "pred_query.npz"
BASE_MODEL_PATH  = TRAIN_DIR / "hybrid_model_best.keras"

USE_AVECSI = True

TARGET_T = 256
TARGET_S = None

BATCH_SIZE = 8
EPOCHS_PHASE1 = 40
EPOCHS_PHASE2 = 50

LR_PHASE1 = 5e-5
LR_PHASE2 = 1e-5

W_PRES  = 0.5
W_CLS   = 1.0
W_AOA   = 1.0
W_DELTA = 0.5
W_FINAL = 1.5

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## Cell 4 - GPU Enable

In [4]:
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print("memory growth warning:", e)

TensorFlow: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Cell 5 - Load Data

In [5]:
pred_support_obj = np.load(PRED_SUPPORT_NPZ, allow_pickle=True)
pred_query_obj   = np.load(PRED_QUERY_NPZ, allow_pickle=True)

print("pred_support n =", len(pred_support_obj["label_id"]))
print("pred_query   n =", len(pred_query_obj["label_id"]))

LABEL_MAP_JSON = BUILD_DIR / "label_map.json"
with open(LABEL_MAP_JSON, "r", encoding="utf-8") as f:
    label_meta = json.load(f)

label_map = label_meta["label_map"]
inv_label_map = {int(k): v for k, v in label_meta["inv_label_map"].items()} if "inv_label_map" in label_meta else None

NUM_CLASSES = len(label_map)
print("NUM_CLASSES =", NUM_CLASSES)
print("label_map =", label_map)

pred_support n = 60
pred_query   n = 526
NUM_CLASSES = 9
label_map = {'Empty': 0, 'LeftDown': 1, 'LeftMid': 2, 'LeftUp': 3, 'MiddleDown': 4, 'MiddleUp': 5, 'RightDown': 6, 'RightMid': 7, 'RightUp': 8}


## Cell 6 - Helper

In [6]:
def np_load_float32(path):
    return np.load(str(path)).astype(np.float32)

def ensure_3d(x):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 2:
        x = x[..., None]
    return x

def resize_to_target(x, target_t, target_s):
    x_tf = tf.convert_to_tensor(x, dtype=tf.float32)
    x_tf = tf.image.resize(x_tf, size=(target_t, target_s), method="bilinear")
    return x_tf.numpy().astype(np.float32)

def zscore_per_sample(x):
    mu = np.mean(x, axis=(0,1), keepdims=True)
    sd = np.std(x, axis=(0,1), keepdims=True) + 1e-6
    return ((x - mu) / sd).astype(np.float32)

def infer_target_s(npz_obj):
    a_amp = np_load_float32(npz_obj["A_amp_paths"][0])
    a_amp = ensure_3d(a_amp)
    return a_amp.shape[1]

if TARGET_S is None:
    TARGET_S = infer_target_s(pred_support_obj)

print("TARGET_T =", TARGET_T)
print("TARGET_S =", TARGET_S)

def build_amp_input_from_row(npz_obj, idx, use_avecsi=True):
    feats = []

    A = ensure_3d(np_load_float32(npz_obj["A_amp_paths"][idx]))
    B = ensure_3d(np_load_float32(npz_obj["B_amp_paths"][idx]))
    feats += [A, B]

    if use_avecsi:
        aavg_path = str(npz_obj["A_ampavg_paths"][idx])
        bavg_path = str(npz_obj["B_ampavg_paths"][idx])

        if aavg_path and bavg_path:
            Aavg = ensure_3d(np_load_float32(aavg_path))
            Bavg = ensure_3d(np_load_float32(bavg_path))

            Tref = A.shape[0]
            if Aavg.shape[0] == 1 and Tref > 1:
                Aavg = np.repeat(Aavg, Tref, axis=0)
            if Bavg.shape[0] == 1 and Tref > 1:
                Bavg = np.repeat(Bavg, Tref, axis=0)

            feats += [Aavg, Bavg]

    T0, S0 = feats[0].shape[:2]
    out = []
    for x in feats:
        if x.shape[0] != T0 or x.shape[1] != S0:
            x = resize_to_target(x, T0, S0)
        out.append(x)

    x = np.concatenate(out, axis=-1)
    x = resize_to_target(x, TARGET_T, TARGET_S)
    x = zscore_per_sample(x)
    return x.astype(np.float32)

def build_phase_input_from_row(npz_obj, idx):
    A = ensure_3d(np_load_float32(npz_obj["A_pha_paths"][idx]))
    B = ensure_3d(np_load_float32(npz_obj["B_pha_paths"][idx]))

    T0 = max(A.shape[0], B.shape[0])
    S0 = max(A.shape[1], B.shape[1])

    if A.shape[:2] != (T0, S0):
        A = resize_to_target(A, T0, S0)
    if B.shape[:2] != (T0, S0):
        B = resize_to_target(B, T0, S0)

    x = np.concatenate([A, B], axis=-1)
    x = resize_to_target(x, TARGET_T, TARGET_S)
    x = zscore_per_sample(x)
    return x.astype(np.float32)

CLASS_CENTER_MAP = {
    "Empty":       [0.0, 0.0],
    "LeftDown":    [2.0, 6.0],
    "LeftMid":     [2.0, 4.0],
    "LeftUp":      [2.0, 2.0],
    "MiddleDown":  [3.0, 0.0],
    "MiddleUp":    [3.0, 2.0],
    "RightDown":   [4.0, 6.0],
    "RightMid":    [4.0, 4.0],
    "RightUp":     [4.0, 2.0],
}

def get_xy_from_label_name(label_name):
    return np.asarray(CLASS_CENTER_MAP[str(label_name)], dtype=np.float32)

def onehot(num_classes, y):
    v = np.zeros((num_classes,), dtype=np.float32)
    v[int(y)] = 1.0
    return v

TARGET_T = 256
TARGET_S = 41


## Cell 7 - Build Amp & Phase

In [7]:
print("NUM_CLASSES =", NUM_CLASSES)

sample_amp = build_amp_input_from_row(pred_support_obj, 0, use_avecsi=USE_AVECSI)
sample_pha = build_phase_input_from_row(pred_support_obj, 0)

print("sample_amp:", sample_amp.shape, sample_amp.dtype)
print("sample_pha:", sample_pha.shape, sample_pha.dtype)

NUM_CLASSES = 9
sample_amp: (256, 41, 8) float32
sample_pha: (256, 41, 4) float32


I0000 00:00:1774122624.485709  534921 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 20718 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9


## Cell 8 - Model Define

In [8]:
class HybridSequence(keras.utils.Sequence):
    def __init__(self, npz_obj, indices, batch_size=8, shuffle=True):
        self.npz_obj = npz_obj
        self.indices = np.array(indices, dtype=np.int64)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        return math.ceil(len(self.indices) / self.batch_size)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        inds = self.indices[idx*self.batch_size:(idx+1)*self.batch_size]

        x_amp = []
        x_pha = []

        y_presence = []
        y_cls = []
        y_xy = []
        y_aoa = []
        y_delta = []
        sample_w = []

        for i in inds:
            amp = build_amp_input_from_row(self.npz_obj, i, use_avecsi=USE_AVECSI)
            pha = build_phase_input_from_row(self.npz_obj, i)

            label_id = int(self.npz_obj["label_id"][i])
            label_name = str(self.npz_obj["label_name"][i])
            presence = int(self.npz_obj["presence"][i])

            xy = get_xy_from_label_name(label_name)
            aoa_xy = xy.copy()
            delta_xy = np.zeros((2,), dtype=np.float32)

            w = 0.25 if presence == 0 else 1.0

            x_amp.append(amp)
            x_pha.append(pha)

            y_presence.append([presence])
            y_cls.append(onehot(NUM_CLASSES, label_id))
            y_xy.append(xy)
            y_aoa.append(aoa_xy)
            y_delta.append(delta_xy)
            sample_w.append(w)

        x = {
            "amp_in": np.stack(x_amp).astype(np.float32),
            "pha_in": np.stack(x_pha).astype(np.float32),
        }

        y = {
            "presence_out": np.stack(y_presence).astype(np.float32),
            "class_out": np.stack(y_cls).astype(np.float32),
            "aoa_xy_out": np.stack(y_aoa).astype(np.float32),
            "amp_delta_out": np.stack(y_delta).astype(np.float32),
            "final_xy_out": np.stack(y_xy).astype(np.float32),
        }

        sw = {
            "presence_out": np.asarray(sample_w, dtype=np.float32),
            "class_out": np.asarray(sample_w, dtype=np.float32),
            "aoa_xy_out": np.asarray(sample_w, dtype=np.float32),
            "amp_delta_out": np.asarray(sample_w, dtype=np.float32),
            "final_xy_out": np.asarray(sample_w, dtype=np.float32),
        }

        return x, y, sw

In [9]:
support_idx = np.arange(len(pred_support_obj["label_id"]))
query_idx   = np.arange(len(pred_query_obj["label_id"]))

support_gen = HybridSequence(pred_support_obj, support_idx, batch_size=BATCH_SIZE, shuffle=True)
query_gen   = HybridSequence(pred_query_obj, query_idx, batch_size=BATCH_SIZE, shuffle=False)

bx, by, bsw = support_gen[0]
print("amp_in:", bx["amp_in"].shape)
print("pha_in:", bx["pha_in"].shape)
for k, v in by.items():
    print(k, v.shape)

amp_in: (8, 256, 41, 8)
pha_in: (8, 256, 41, 4)
presence_out (8, 1)
class_out (8, 9)
aoa_xy_out (8, 2)
amp_delta_out (8, 2)
final_xy_out (8, 2)


## Cell 9 - Build Model

In [10]:
losses = {
    "presence_out": keras.losses.BinaryCrossentropy(),
    "class_out": keras.losses.CategoricalCrossentropy(),
    "aoa_xy_out": keras.losses.Huber(),
    "amp_delta_out": keras.losses.Huber(),
    "final_xy_out": keras.losses.Huber(),
}

loss_weights = {
    "presence_out": W_PRES,
    "class_out": W_CLS,
    "aoa_xy_out": W_AOA,
    "amp_delta_out": W_DELTA,
    "final_xy_out": W_FINAL,
}

metrics = {
    "presence_out": [keras.metrics.BinaryAccuracy(name="acc")],
    "class_out": [keras.metrics.CategoricalAccuracy(name="acc")],
    "aoa_xy_out": [keras.metrics.MeanAbsoluteError(name="mae")],
    "amp_delta_out": [keras.metrics.MeanAbsoluteError(name="mae")],
    "final_xy_out": [keras.metrics.MeanAbsoluteError(name="mae")],
}

base_model = keras.models.load_model(
    BASE_MODEL_PATH,
    compile=False,
    safe_mode=False
)

print("Loaded base model:", BASE_MODEL_PATH)
base_model.summary()

Loaded base model: /home/tonyliao/Location_AMP/hybrid_train_runs/hybrid_model_best.keras


Model: "hybrid_aoa_amp_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ amp_in (InputLayer) │ (None, 256, 41,   │          0 │ -                 │
│                     │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ amp_ssl_encoder     │ (None, 256)       │  1,308,160 │ amp_in[0][0]      │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pha_in (InputLayer) │ (None, 256, 41,   │          0 │ -                 │
│                     │ 4)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ amp_feat (Dense)    │ (None, 128)       │     32,896 │ amp_ssl_encoder[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ phase_geometry_enc… │ (None, 128)       │  1,274,112 │ pha_in[0][0]      │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 256)       │          0 │ amp_feat[0][0],   │
│ (Concatenate)       │                   │            │ phase_geometry_e… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     32,896 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ class_out (Dense)   │ (None, 9)         │      1,161 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ presence_out        │ (None, 1)         │        129 │ amp_feat[0][0]    │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 128)       │     16,512 │ amp_feat[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 266)       │          0 │ amp_feat[0][0],   │
│ (Concatenate)       │                   │            │ phase_geometry_e… │
│                     │                   │            │ presence_out[0][… │
│                     │                   │            │ class_out[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 2)         │        258 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 64)        │     17,088 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ amp_delta_out       │ (None, 2)         │          0 │ dense_4[0][0]     │
│ (Rescaling)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     16,512 │ phase_geometry_e… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ delta_gate (Dense)  │ (None, 1)         │         65 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ aoa_xy_out (Dense)  │ (None, 2)         │        258 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 2)         │          0 │ amp_delta_out[0]… │
│                     │                   │            │ delta_gate[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ final_xy_out (Add)  │ (None, 2)         │          0 │ aoa_xy_out[0][0], │
│                     │                   │            │ multiply[0][0]  

 Total params: 2,700,047 (10.30 MB)

 Trainable params: 1,521,551 (5.80 MB)

 Non-trainable params: 1,178,496 (4.50 MB)

## Cell 10 - Compile Defined

In [11]:
def compile_model(model, lr):
    opt = keras.optimizers.Adam(learning_rate=lr)
    model.compile(
        optimizer=opt,
        loss=losses,
        loss_weights=loss_weights,
        metrics=metrics,
    )

## Cell 11 - Fine Tuning Phase 1

In [12]:
# Phase 1: freeze almost everything, tune top heads
for layer in base_model.layers:
    layer.trainable = False

for name in [
    "presence_out",
    "class_out",
    "aoa_xy_out",
    "amp_delta_out",
    "final_xy_out",
]:
    try:
        base_model.get_layer(name).trainable = True
    except Exception:
        pass

for layer in base_model.layers[-20:]:
    layer.trainable = True

compile_model(base_model, LR_PHASE1)

print("Phase 1 trainable layers:")
for layer in base_model.layers[-15:]:
    print(layer.name, layer.trainable)

callbacks_phase1 = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(OUT_DIR / "hybrid_transfer_phase1_best.keras"),
        monitor="val_final_xy_out_mae",
        save_best_only=True,
        mode="min",
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_final_xy_out_mae",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        mode="min",
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_final_xy_out_mae",
        patience=8,
        mode="min",
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.CSVLogger(str(OUT_DIR / "transfer_phase1_log.csv")),
]

history_phase1 = base_model.fit(
    support_gen,
    validation_data=query_gen,
    epochs=EPOCHS_PHASE1,
    callbacks=callbacks_phase1,
    verbose=1,
)


Phase 1 trainable layers:
phase_geometry_encoder True
concatenate True
dense_1 True
class_out True
presence_out True
dense_3 True
concatenate_1 True
dense_4 True
dense_5 True
amp_delta_out True
dense_2 True
delta_gate True
aoa_xy_out True
multiply True
final_xy_out True
Epoch 1/40


/home/tonyliao/tensorflow_env/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
I0000 00:00:1774122628.132616  535063 service.cc:152] XLA service 0x7f52f8043f90 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774122628.132629  535063 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2026-03-21 19:50:28.230519: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1774122628.875082  535063 cuda_dnn.cc:529] Loaded cuDNN version 90300
2026-03-21 19:50:29.973402: I external/loc

1/8 ━━━━━━━━━━━━━━━━━━━━ 1:31 13s/step - amp_delta_out_loss: 0.0560 - amp_delta_out_mae: 0.3331 - aoa_xy_out_loss: 0.7619 - aoa_xy_out_mae: 2.4567 - class_out_acc: 0.1250 - class_out_loss: 2.2740 - final_xy_out_loss: 0.7613 - final_xy_out_mae: 2.4580 - loss: 4.8605 - presence_out_acc: 0.5000 - presence_out_loss: 1.3094

I0000 00:00:1774122638.319593  535063 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-03-21 19:50:40.457111: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_4716', 8 bytes spill stores, 8 bytes spill loads



7/8 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - amp_delta_out_loss: 0.0646 - amp_delta_out_mae: 0.3310 - aoa_xy_out_loss: 0.8740 - aoa_xy_out_mae: 2.2086 - class_out_acc: 0.1263 - class_out_loss: 2.4036 - final_xy_out_loss: 0.8738 - final_xy_out_mae: 2.2111 - loss: 4.8613 - presence_out_acc: 0.6107 - presence_out_loss: 0.8440

2026-03-21 19:50:51.070418: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_363', 8 bytes spill stores, 8 bytes spill loads




Epoch 1: val_final_xy_out_mae improved from inf to 1.04158, saving model to /home/tonyliao/Location_AMP/hybrid_transfer_runs/hybrid_transfer_phase1_best.keras
8/8 ━━━━━━━━━━━━━━━━━━━━ 28s 2s/step - amp_delta_out_loss: 0.0651 - amp_delta_out_mae: 0.3333 - aoa_xy_out_loss: 0.8367 - aoa_xy_out_mae: 2.1263 - class_out_acc: 0.1538 - class_out_loss: 2.2925 - final_xy_out_loss: 0.8368 - final_xy_out_mae: 2.1292 - loss: 4.6457 - presence_out_acc: 0.6231 - presence_out_loss: 0.8154 - val_amp_delta_out_loss: 0.0776 - val_amp_delta_out_mae: 0.3655 - val_aoa_xy_out_loss: 0.3077 - val_aoa_xy_out_mae: 1.0341 - val_class_out_acc: 0.6331 - val_class_out_loss: 0.7224 - val_final_xy_out_loss: 0.3102 - val_final_xy_out_mae: 1.0416 - val_loss: 1.8093 - val_presence_out_acc: 0.7586 - val_presence_out_loss: 0.5519 - learning_rate: 5.0000e-05
Epoch 2/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - amp_delta_out_loss: 0.0499 - amp_delta_out_mae: 0.3149 - aoa_xy_out_loss: 0.3309 - aoa_xy_out_mae: 1.1156 - class_ou

## Cell 12 - Fine Tuning Phase 2

In [13]:
# Phase 2: unfreeze more layers for low-LR adaptation
for layer in base_model.layers:
    layer.trainable = True

# optional: keep earliest layers frozen
for layer in base_model.layers[:10]:
    layer.trainable = False

compile_model(base_model, LR_PHASE2)

print("Phase 2 trainable summary:")
for layer in base_model.layers[:15]:
    print(layer.name, layer.trainable)

callbacks_phase2 = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(OUT_DIR / "hybrid_transfer_best.keras"),
        monitor="val_final_xy_out_mae",
        save_best_only=True,
        mode="min",
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_final_xy_out_mae",
        factor=0.5,
        patience=4,
        min_lr=1e-7,
        mode="min",
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_final_xy_out_mae",
        patience=8,
        mode="min",
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.CSVLogger(str(OUT_DIR / "transfer_phase2_log.csv")),
]

history_phase2 = base_model.fit(
    support_gen,
    validation_data=query_gen,
    epochs=EPOCHS_PHASE2,
    callbacks=callbacks_phase2,
    verbose=1,
)

Phase 2 trainable summary:
amp_in False
amp_ssl_encoder False
pha_in False
amp_feat False
phase_geometry_encoder False
concatenate False
dense_1 False
class_out False
presence_out False
dense_3 False
concatenate_1 True
dense_4 True
dense_5 True
amp_delta_out True
dense_2 True
Epoch 1/50
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 388ms/step - amp_delta_out_loss: 0.0133 - amp_delta_out_mae: 0.1822 - aoa_xy_out_loss: 0.0010 - aoa_xy_out_mae: 0.0969 - class_out_acc: 1.0000 - class_out_loss: 7.5356e-04 - final_xy_out_loss: 0.0010 - final_xy_out_mae: 0.0971 - loss: 0.0108 - presence_out_acc: 1.0000 - presence_out_loss: 0.0016     
Epoch 1: val_final_xy_out_mae improved from inf to 0.10293, saving model to /home/tonyliao/Location_AMP/hybrid_transfer_runs/hybrid_transfer_best.keras
8/8 ━━━━━━━━━━━━━━━━━━━━ 10s 929ms/step - amp_delta_out_loss: 0.0136 - amp_delta_out_mae: 0.1818 - aoa_xy_out_loss: 0.0011 - aoa_xy_out_mae: 0.0963 - class_out_acc: 1.0000 - class_out_loss: 8.0734e-04 - final_xy_out_loss: 0.0011 -

## Cell 13 - Save Model

In [14]:
base_model.save(OUT_DIR / "hybrid_transfer_final.keras")

history_all = {
    "phase1": history_phase1.history,
    "phase2": history_phase2.history,
}

with open(OUT_DIR / "transfer_history.json", "w", encoding="utf-8") as f:
    json.dump(history_all, f, indent=2)

print("Saved transfer model to:", OUT_DIR)

best_model = keras.models.load_model(
    OUT_DIR / "hybrid_transfer_best.keras",
    compile=False,
    safe_mode=False
)
print("Loaded transfer best model.")
best_model.summary()

Saved transfer model to: /home/tonyliao/Location_AMP/hybrid_transfer_runs
Loaded transfer best model.


Model: "hybrid_aoa_amp_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ amp_in (InputLayer) │ (None, 256, 41,   │          0 │ -                 │
│                     │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ amp_ssl_encoder     │ (None, 256)       │  1,308,160 │ amp_in[0][0]      │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pha_in (InputLayer) │ (None, 256, 41,   │          0 │ -                 │
│                     │ 4)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ amp_feat (Dense)    │ (None, 128)       │     32,896 │ amp_ssl_encoder[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ phase_geometry_enc… │ (None, 128)       │  1,274,112 │ pha_in[0][0]      │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 256)       │          0 │ amp_feat[0][0],   │
│ (Concatenate)       │                   │            │ phase_geometry_e… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     32,896 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ class_out (Dense)   │ (None, 9)         │      1,161 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ presence_out        │ (None, 1)         │        129 │ amp_feat[0][0]    │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 128)       │     16,512 │ amp_feat[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 266)       │          0 │ amp_feat[0][0],   │
│ (Concatenate)       │                   │            │ phase_geometry_e… │
│                     │                   │            │ presence_out[0][… │
│                     │                   │            │ class_out[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 2)         │        258 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 64)        │     17,088 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ amp_delta_out       │ (None, 2)         │          0 │ dense_4[0][0]     │
│ (Rescaling)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     16,512 │ phase_geometry_e… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ delta_gate (Dense)  │ (None, 1)         │         65 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ aoa_xy_out (Dense)  │ (None, 2)         │        258 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 2)         │          0 │ amp_delta_out[0]… │
│                     │                   │            │ delta_gate[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ final_xy_out (Add)  │ (None, 2)         │          0 │ aoa_xy_out[0][0], │
│                     │                   │            │ multiply[0][0]  

 Total params: 2,700,047 (10.30 MB)

 Trainable params: 34,181 (133.52 KB)

 Non-trainable params: 2,665,866 (10.17 MB)

## Cell 14 - Evaluation

In [15]:
def predict_one(npz_obj, idx, model):
    amp = build_amp_input_from_row(npz_obj, idx, use_avecsi=USE_AVECSI)
    pha = build_phase_input_from_row(npz_obj, idx)

    x = {
        "amp_in": np.expand_dims(amp, axis=0),
        "pha_in": np.expand_dims(pha, axis=0),
    }

    pred = model.predict(x, verbose=0)

    out = {
        "presence": float(pred["presence_out"][0,0]),
        "class_prob": pred["class_out"][0],
        "class_id": int(np.argmax(pred["class_out"][0])),
        "aoa_xy": pred["aoa_xy_out"][0],
        "amp_delta": pred["amp_delta_out"][0],
        "final_xy": pred["final_xy_out"][0],
    }
    return out

example = predict_one(pred_query_obj, 0, best_model)
for k, v in example.items():
    print(k, v)

def evaluate_query(npz_obj, model, max_n=None):
    n = len(npz_obj["label_id"]) if max_n is None else min(len(npz_obj["label_id"]), max_n)

    y_true = []
    y_pred = []
    cls_true = []
    cls_pred = []

    for i in range(n):
        label_name = str(npz_obj["label_name"][i])
        true_xy = get_xy_from_label_name(label_name)

        pred = predict_one(npz_obj, i, model)

        y_true.append(true_xy)
        y_pred.append(pred["final_xy"])
        cls_true.append(int(npz_obj["label_id"][i]))
        cls_pred.append(pred["class_id"])

    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)

    err = np.linalg.norm(y_true - y_pred, axis=1)
    cls_acc = np.mean(np.asarray(cls_true) == np.asarray(cls_pred))

    return {
        "n": int(n),
        "mean_xy_error": float(np.mean(err)),
        "median_xy_error": float(np.median(err)),
        "class_acc": float(cls_acc),
    }

metrics_query = evaluate_query(pred_query_obj, best_model, max_n=None)
print(metrics_query)

presence 0.998313307762146
class_prob [6.5045012e-04 8.9081546e-04 3.6455920e-05 9.9669468e-01 8.8598119e-04
 1.4040066e-04 1.3954593e-04 4.0220455e-04 1.5953281e-04]
class_id 3
aoa_xy [1.8425078 2.0197139]
amp_delta [-0.3847801   0.16584416]
final_xy [1.8332119 2.0237205]
{'n': 526, 'mean_xy_error': 0.1603974848985672, 'median_xy_error': 0.1403806358575821, 'class_acc': 1.0}


## Cell 15 - Summary

In [16]:
summary = {
    "pred_support_npz": str(PRED_SUPPORT_NPZ),
    "pred_query_npz": str(PRED_QUERY_NPZ),
    "base_model_path": str(BASE_MODEL_PATH),
    "target_t": TARGET_T,
    "target_s": TARGET_S,
    "batch_size": BATCH_SIZE,
    "epochs_phase1": EPOCHS_PHASE1,
    "epochs_phase2": EPOCHS_PHASE2,
    "lr_phase1": LR_PHASE1,
    "lr_phase2": LR_PHASE2,
    "weights": {
        "presence": W_PRES,
        "class": W_CLS,
        "aoa": W_AOA,
        "delta": W_DELTA,
        "final": W_FINAL,
    },
    "query_metrics": metrics_query,
    "seed": SEED,
}

with open(OUT_DIR / "transfer_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Summary saved.")

Summary saved.


In [ ]:
import tensorflow as tf
tf.keras.backend.clear_session()
#Restart the kernel to free memory
import IPython
app = IPython.get_ipython()
app.kernel.do_shutdown(True)  # True = restart, False = shutdown

{'status': 'ok', 'restart': True}

: 